<a href="https://colab.research.google.com/github/faith-amanze/assignment1/blob/main/Copy_of_w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [ ]:
# CONTRACT — Unit of analysis + time window:
# One row = one page's observed performance for one client on one calendar day
# (grain: report_date + client_hash_id + content_hash_id) in fact_content_daily_performance.
# Time window for this notebook: month = '2026-03' (mid-panel — never the sealed final month).

con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_grain
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ distinct_grain │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        9841378 │
└────────────┴────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# FIELD CLASSIFICATION

# FEATURES (knowable at decision time):
#   impressions        - observed daily GSC signal, already recorded history
#   avg_position        - observed daily ranking, already recorded history
#   ctr                 - derived from clicks/impressions, both already observed
#   content_age_days    - static content attribute (creation date is fixed)
#   word_count           - static content attribute from dim_content

# LABEL / PROXY:
#   is_declining  - derived from comparing first-half vs second-half impressions
#                   within the SAME March window (a within-window proxy, not yet
#                   a true past->future label)

# CONTEXT (background, not features, not labels):
#   client_hash_id, content_hash_id  - join keys only, carry no signal themselves
#   is_active, has_gsc_access, has_ga4_access  - used to filter/validate the
#                   slice, not fed into the model
#   ga4_data_available  - used for the availability check (IS TRUE filter),
#                   not a model feature

# EXCLUDED:
#   rows where is_active = FALSE or has_gsc_access = FALSE at the client level
#   -- why: a refresh-review queue should never surface pages for clients who
#   are inactive or have no search data connected, even if stray rows exist
#   health_score, priority_score, action_type, or any other FlyRank product
#   decision flag -- why: these are not in the dataset at all, and even if
#   rebuilt, using them as a feature or label would just teach the model to
#   copy an existing rule instead of discovering real signal (circular result)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# QUERY 2 — row count and date span for the March slice
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [ ]:
# QUERY 3 — availability, filtered with IS TRUE
con.sql(f"""
    SELECT COUNT(*) AS rows_before_filter
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
""").show()

con.sql(f"""
    SELECT COUNT(*) AS rows_after_filter
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
      AND ga4_data_available IS TRUE
""").show()

┌────────────────────┐
│ rows_before_filter │
│       int64        │
├────────────────────┤
│            9841378 │
└────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┐
│ rows_after_filter │
│       int64       │
├───────────────────┤
│            413966 │
└───────────────────┘



In [ ]:
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1
""").show(max_rows=100)

┌──────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name        │ column_type │  null   │   key   │ default │  extra  │
│         varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date              │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available       │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gs

In [ ]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 1").show(max_rows=100)

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [ ]:
import time

def run_query(sql, tries=4, wait=8):
    for attempt in range(tries):
        try:
            return con.sql(sql)
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            if attempt < tries - 1:
                time.sleep(wait)
            else:
                raise

In [ ]:
half = run_query(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# FIVE FEATURES — built from the same March 2026 window

features = run_query(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        AVG(gsc_avg_position) AS avg_position_month,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
             ELSE 0 END AS ctr_month
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

content = run_query(f"SELECT content_hash_id, word_count, content_created_date FROM read_parquet('{rel}/dim_content.parquet')").df()

import pandas as pd
features = features.merge(content, on="content_hash_id", how="left")
features["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(features["content_created_date"])).dt.days

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_month,avg_position_month,ctr_month,word_count,content_created_date,content_age_days
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,4.394234,0.001754,<NA>,2025-02-28,396
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,2.714744,0.000000,<NA>,2025-02-28,396
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,6.481453,0.000000,<NA>,2025-02-28,396
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.320337,0.004222,<NA>,2025-02-28,396
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,4.459107,0.005776,2475,2025-02-28,396


In [ ]:
half["is_declining"] = (half["impr_second_half"] < half["impr_first_half"]).astype(int)

features_labeled = features.merge(
    half[["client_hash_id","content_hash_id","impr_first_half","impr_second_half","is_declining"]],
    on=["client_hash_id","content_hash_id"]
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# HONEST model — five real features only
X_honest = features_labeled[["impressions_month","avg_position_month","ctr_month","content_age_days","word_count"]].fillna(0)
y = features_labeled["is_declining"]

model = LogisticRegression(max_iter=1000).fit(X_honest, y)
honest_score = roc_auc_score(y, model.predict_proba(X_honest)[:,1])
print("Honest AUC:", honest_score)

# THE TRAP — add a column derived directly from the label
features_labeled["leak_half_diff"] = features_labeled["impr_second_half"] - features_labeled["impr_first_half"]

X_leaky = features_labeled[["impressions_month","avg_position_month","ctr_month","content_age_days","word_count","leak_half_diff"]].fillna(0)
leaky_model = LogisticRegression(max_iter=1000).fit(X_leaky, y)
leaky_score = roc_auc_score(y, leaky_model.predict_proba(X_leaky)[:,1])
print("Leaky AUC (jumps toward 1.0):", leaky_score)

# Delete the leaky column, keep the honest number
features_labeled = features_labeled.drop(columns=["leak_half_diff"])
print("Final honest AUC kept:", honest_score)

Honest AUC: 0.71657184048735
Leaky AUC (jumps toward 1.0): 1.0
Final honest AUC kept: 0.71657184048735


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# LIMITATION

# This slice is drawn from an unbalanced panel: clients have different tracking
# start dates (dim_clients.gsc_data_start / ga4_data_start), so March 2026
# contains uneven history depth per client. Only ~4.2% of March rows had
# ga4_data_available = TRUE (413,966 of 9,841,378), meaning any GA4-derived
# signal (sessions, engagement) would only apply to a small, possibly
# non-representative fraction of this slice -- a client whose GA4 tracking
# started mid-month, or never at all, will look artificially thin on
# engagement signals, which could bias any decline/opportunity proxy built
# on this window toward clients with longer, more complete tracking history.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.